<a href="https://colab.research.google.com/github/binghan1123/1142-programming-language/blob/main/HW2_%E6%88%90%E7%B8%BE%E4%B8%80%E6%9C%AC%E9%80%9A%E6%9C%80%E7%B5%82%E7%89%88.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q google-generativeai

In [2]:
import gradio as gr
import pandas as pd
from google.colab import auth
from google.auth import default

# -*- coding: utf-8 -*-
import gspread
from datetime import datetime
import google.generativeai as genai
import os
import json

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [3]:
from google.colab import userdata
from google import genai

# 從 Colab Secrets 中獲取 API 金鑰
api_key = userdata.get('41371209H')

# 使用獲取的金鑰配置 genai
client = genai.Client(api_key=api_key)

MODEL_ID = 'gemini-2.5-flash-lite'

In [4]:
response = client.models.generate_content(
    model="gemini-2.5-flash-lite", contents="Explain how AI works in a few words"
)
print(response.text)

AI learns from data to make decisions or predictions.


In [5]:
response = client.models.generate_content(
    model = MODEL_ID, contents="Explain how AI works in a few words"
)
print(response.text)

AI learns from data to make decisions and perform tasks.


In [6]:
SHEET_URL = "https://docs.google.com/spreadsheets/d/1yefD7L8LSK27T-B_V4PWyk6tEUl5IGUCs9Lw8SbCGF0/edit?usp=sharing"
WORKSHEET_NAME = "工作表2"

REQUIRED_COLUMNS = ["日期", "科目", "作業成績","等第"]

_auth_done = False
_gc = None
_ws = None

In [7]:
# --- 主要功能區塊 ---
def get_user_grades():
    """
    透過終端機輸入學生成績，直到使用者輸入 'end' 結束。
    """
    print("--- 準備輸入成績。輸入 'end' 來停止。---")
    grades = []
    while True:
        subject = input("請輸入科目（或輸入 'end' 停止）：")
        if subject.lower() == 'end':
            break

        grade = input(f"請輸入 {subject} 的成績：")
        try:
            grade = int(grade)
        except ValueError:
            print("成績必須是數字。請重新輸入。")
            continue

        today = datetime.now().strftime('%Y-%m-%d')
        grades.append([today, subject, grade])
        print(f"已記錄：日期: {today}, 科目: {subject}, 成績: {grade}\n")

    return grades

In [8]:
def get_ai_summary(grades):
    """
    呼叫 Gemini 模型來生成成績摘要與常見迷思。
    """
    # 準備給 AI 的提示
    prompt_text = "以下是學生的成績列表，請幫我根據這些成績，產出一個簡單的摘要與常見迷思整理（不評分，只做總結）。\n\n"
    for record in grades:
        date, subject, grade = record
        prompt_text += f"日期：{date}, 科目：{subject}, 成績：{grade}\n"

    print("\n--- 正在呼叫 AI 模型生成摘要... ---")
    try:
        response = client.models.generate_content(model=MODEL_ID, contents=prompt_text)

        summary = ""
        # Prioritize response.text as it's the most straightforward way to get the text output
        if hasattr(response, 'text') and response.text:
            summary = response.text
        elif hasattr(response, 'candidates') and response.candidates:
            # Fallback to manual parsing if response.text didn't work or for structured output
            # Check if the first candidate has content and parts
            if hasattr(response.candidates[0], 'content') and hasattr(response.candidates[0].content, 'parts'):
                for part in response.candidates[0].content.parts:
                    if hasattr(part, 'text'):
                        summary += part.text

        if not summary:
            # If summary is still empty, something went wrong or no text was generated
            summary = f"AI 摘要生成失敗：未能從模型獲得有效回應或回應內容為空。原始回應類型: {type(response)}, 原始回應: {response}"
            print(f"警告：AI 模型回應無有效內容。原始回應: {response}")

        return summary
    except Exception as e:
        print(f"呼叫 AI 時發生錯誤：{e}")
        return f"AI 摘要生成失敗。錯誤訊息: {e}"

In [13]:
def get_grade(score):
    if score >= 90: return 'A+'
    elif score >= 85: return 'A'
    elif score >= 80: return 'A-'
    elif score >= 77: return 'B+'
    elif score >= 73: return 'B'
    elif score >= 70: return 'B-'
    elif score >= 67: return 'C+'
    elif score >= 63: return 'C'
    elif score >= 60: return 'C-'
    elif score >= 50: return 'D'
    elif score >= 1: return 'E'
    else: return 'X'

def main():
    """
    主程式流程：輸入成績 -> 獲取 AI 摘要 -> 寫入 Google Sheet。
    """
    try:
        # 1. Google Sheet 身份驗證
        auth.authenticate_user()

        creds, _ = default()
        gc = gspread.authorize(creds)

        sh = gc.open_by_url(SHEET_URL)
        ws = sh.worksheet(WORKSHEET_NAME)

        print("--- Google Sheet 連線成功。---")

        # 檢查並確保標題列存在
        current_data = ws.get_all_values()

        # 先預設一個標籤，決定是否要新增標題
        need_header = False

        if not current_data:
            # 情況 1：表單完全是空的（連格子都沒有）
            need_header = True
        elif len(current_data[0]) == 0 or current_data[0][0] != "日期":
            # 情況 2：表單有格子但第一列是空的，或者第一格的字不是「日期」
            need_header = True

        # 根據上面的判斷結果來執行動作
        if need_header:
            ws.insert_row(REQUIRED_COLUMNS, 1)
            print(f"--- 試算表目前無標題，已自動生成標題列：{REQUIRED_COLUMNS} ---")
        else:
            print("--- 標題列已存在，不重複生成。---")

        # 2. 獲取使用者輸入的成績
        # get_user_grades() 返回格式為 [[date, subject, grade]]
        raw_new_grades = get_user_grades()

        if not raw_new_grades:
            print("沒有輸入任何成績，程式結束。")
            return

        # 處理 raw_new_grades，加入「等第」
        processed_grades = []
        for record in raw_new_grades:
            date, subject, score = record
            grade_level = get_grade(score)
            processed_grades.append([date, subject, score, grade_level])

        # 3. 將新成績寫入 Google Sheet
        ws.append_rows(processed_grades)
        print("\n--- 成績已成功寫入 Google Sheet。---")

        # 4. 獲取 AI 摘要並寫入 Google Sheet
        # 準備傳給 get_ai_summary 的資料，它預期格式為 [日期, 科目, 成績]
        ai_summary_input = [[record[0], record[1], record[2]] for record in processed_grades]
        summary = get_ai_summary(ai_summary_input)

        # 將 AI 摘要作為一個新的行寫入，日期在第一欄，'AI 摘要'在第二欄，完整內容在第三欄
        summary_entry = [
            datetime.now().strftime('%Y-%m-%d'),
            'AI 摘要',
            summary
        ]
        ws.append_row(summary_entry)

        print("\n--- AI 摘要已成功寫入 Google Sheet。---")
        print("以下是 AI 生成的摘要內容：")
        print("-" * 50)
        print(summary)
        print("-" * 50)

    except gspread.exceptions.APIError as e:
        print(f"Google Sheets API 錯誤：{e.response.text}")
        print("請確認：")
        print("1. 您的服務帳戶金鑰檔案正確且未過期。")
        print("2. 您已將服務帳戶的 Email 地址（在 JSON 檔案中）分享給 Google Sheet，並給予編輯權限。")
    except Exception as e:
        print(f"發生未預期的錯誤：{e}")

if __name__ == "__main__":
    main()

--- Google Sheet 連線成功。---
--- 試算表目前無標題，已自動生成標題列：['日期', '科目', '作業成績', '等第'] ---
--- 準備輸入成績。輸入 'end' 來停止。---
請輸入科目（或輸入 'end' 停止）：英文
請輸入 英文 的成績：90
已記錄：日期: 2026-04-20, 科目: 英文, 成績: 90

請輸入科目（或輸入 'end' 停止）：end

--- 成績已成功寫入 Google Sheet。---

--- 正在呼叫 AI 模型生成摘要... ---

--- AI 摘要已成功寫入 Google Sheet。---
以下是 AI 生成的摘要內容：
--------------------------------------------------
好的，針對您提供的學生英文成績，我將產出一個簡單的摘要與常見迷思整理：

---

### **學生英文成績摘要**

**日期：** 2026-04-20
**科目：** 英文
**成績：** 90

**總結：**
根據所提供的單一數據點，該名學生的英文成績為90分。這是一個相當不錯的分數，顯示在本次英文測驗中，學生展現了良好的學習成效。

---

### **與高分相關的常見迷思整理**

雖然90分是一個亮眼的成績，但在解讀時，我們也需要留意一些常見的迷思，避免過度或不準確的判斷：

1.  **迷思一：90分就代表「完全精熟」或「滿分」等級。**
    *   **事實：** 90分通常代表「非常優秀」或「接近滿分」，但並不一定代表學生在該科目上已達到了所有可能性的最高水平，或者完全沒有任何可以進步的空間。滿分可能更高（例如100分），或者該科目還有更深入的知識、技能或應用層面。

2.  **迷思二：90分就代表學生「英文能力極佳」，無需再額外加強。**
    *   **事實：** 這是一次單次的測驗結果。學生的整體英文能力涵蓋聽、說、讀、寫、文法、詞彙等多個面向。僅憑一次90分，無法斷定其在所有面向都達到頂尖水準。例如，口語或聽力可能仍有提升空間，或者是在特定類型的題目上表現較好。

3.  **迷思三：90分表示學習過程「毫不費力」或「沒有遇到困難」。**
    *   **事實：** 高分並不等於沒有付出努力或沒有遇到挑戰。學生可

In [10]:
import plotly.express as px
import pandas as pd
from datetime import datetime

def get_grade(score):
    if score >= 90: return 'A+'
    elif score >= 85: return 'A'
    elif score >= 80: return 'A-'
    elif score >= 77: return 'B+'
    elif score >= 73: return 'B'
    elif score >= 70: return 'B-'
    elif score >= 67: return 'C+'
    elif score >= 63: return 'C'
    elif score >= 60: return 'C-'
    elif score >= 50: return 'D'
    elif score >= 1: return 'E'
    else: return 'X'

def gradio_get_ai_summary(input_text: str):
    try:
        today = datetime.now().strftime('%Y-%m-%d')
        new_records_for_sheet = []
        subjects = []
        scores = []
        grades = []

        lines = input_text.strip().split('\n')
        for line in lines:
            parts = line.split()
            if len(parts) >= 2:
                subject = parts[0]
                try:
                    score = int(parts[1])
                except ValueError:
                    continue
                grade = get_grade(score)
                # 準備寫入 Google Sheet 的資料：日期, 科目, 分數, 等第
                new_records_for_sheet.append([today, subject, score, grade])
                subjects.append(subject)
                scores.append(score)
                grades.append(grade)

        if not new_records_for_sheet:
            return "請輸入正確格式 (科目 分數)", None

        # 1. 呼叫 AI 摘要 (傳入基礎成績資料)
        ai_input_data = [[r[0], r[1], r[2]] for r in new_records_for_sheet]
        summary = get_ai_summary(ai_input_data)

        # 2. 建立 Plotly 圖表
        df_plot = pd.DataFrame({'科目': subjects, '分數': scores, '等第': grades})
        fig = px.bar(df_plot, x='科目', y='分數', text='等第', color='分數',
                     color_continuous_scale='Viridis', range_y=[0, 100])
        fig.update_traces(textposition='outside')

        # 3. 寫入 Google Sheet
        auth.authenticate_user()
        creds, _ = default()
        gc = gspread.authorize(creds)
        sh = gc.open_by_url(SHEET_URL)
        ws = sh.worksheet(WORKSHEET_NAME)

        # 檢查並更新標題列以包含「等第」
        headers = ws.row_values(1)
        if "日期" not in headers:
            ws.update_cell(1, 1, "日期")
        if "科目" not in headers:
            ws.update_cell(1, 2, "科目")
        if "分數" not in headers:
            ws.update_cell(1, 3, "分數")
        if "等第" not in headers:
            ws.update_cell(1, 4, "等第")

        # 寫入包含等第的成績資料
        ws.append_rows(new_records_for_sheet)
        # 寫入摘要記錄
        ws.append_row([today, 'AI 摘要', summary])

        return f"【成功記錄 {len(new_records_for_sheet)} 筆至 Google Sheet】\n\n{summary}", fig
    except Exception as e:
        return f"發生錯誤：{str(e)}", None

## Gradio AI 成績摘要介面

您可以使用下面的 Gradio 介面，輸入單一科目的成績，並即時獲得 AI 生成的摘要和常見迷思整理。

In [14]:
custom_css = """
#centered_input span { display: block; text-align: center; width: 100%; }
#centered_output span { display: block; text-align: center; width: 100%; }
"""

with gr.Blocks(css=custom_css) as iface:
    gr.Markdown("<center><h1>成績摘要生成器</h1></center>")
    gr.Markdown("一次輸入多個科目與成績，獲取 AI 總結並自動紀錄至試算表")

    with gr.Row():
        with gr.Column(scale=1):
            input_text = gr.Textbox(
                label="成績輸入",
                placeholder="格式：科目 成績 (每行一筆，科目跟成績之間需有空格)\n例如：\n國文 85\n數學 92",
                lines=10,
                elem_id="centered_input"
            )
            btn_generate = gr.Button("生成摘要與分析圖表", variant="primary")
            btn_clear = gr.Button("清空資料", variant="secondary")

        with gr.Column(scale=1):
            output_text = gr.Textbox(
                label="AI 摘要結果",
                lines=10,
                max_lines=10,
                elem_id="centered_output"
            )
            output_plot = gr.Plot(label="成績分布圖")

    btn_generate.click(
        fn=gradio_get_ai_summary,
        inputs=input_text,
        outputs=[output_text, output_plot]
    )
    btn_clear.click(fn=lambda: ["", "", None], inputs=None, outputs=[input_text, output_text, output_plot])

iface.launch(debug=True, share=True)

/tmp/ipykernel_995/615090286.py:6: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css) as iface:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://fff285545bc5ac9170.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



--- 正在呼叫 AI 模型生成摘要... ---
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://fff285545bc5ac9170.gradio.live
